# MOT16 Data Visualization

This notebook contains the visualization side of the project:
- inspect MOT16 metadata and GT structure
- render GT annotations into GIFs
- check dataset quality across train and test
- run latest YOLO-based person tracking on MOT16 sequences

This keeps visualization separate from the ingestion notebook.


In [1]:
# Setup: imports, paths, GT names, and helpers
from pathlib import Path
import configparser

import cv2
import pandas as pd
from PIL import Image, ImageDraw, ImageFont
import imageio.v2 as imageio
import ipywidgets as widgets
from IPython.display import Image as IPyImage, clear_output, display
from ultralytics import YOLO

cwd = Path.cwd().resolve()
if (cwd / "MOT16").exists():
    PROJECT_ROOT = cwd
elif (cwd.parent / "MOT16").exists():
    PROJECT_ROOT = cwd.parent
else:
    raise FileNotFoundError("Could not find MOT16 folder from current working directory.")

MOT16_ROOT = PROJECT_ROOT / "MOT16"
OUTPUT_DIR = PROJECT_ROOT / "output" / "jupyter-notebook"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

GT_CLASS_NAMES = {
    1: "pedestrian",
    2: "person_on_vehicle",
    3: "car",
    4: "bicycle",
    5: "motorbike",
    6: "non_mot_vehicle",
    7: "static_person",
    8: "distractor",
    9: "occluder",
    10: "occluder_ground",
    11: "occluder_full",
    12: "reflection",
    13: "crowd",
}

GT_CLASS_COLORS = {
    1: (0, 255, 0),
    2: (255, 165, 0),
    3: (255, 0, 0),
    4: (0, 200, 255),
    5: (255, 0, 255),
    6: (180, 0, 255),
    7: (0, 255, 255),
    8: (255, 215, 0),
    9: (160, 160, 160),
    10: (120, 120, 120),
    11: (80, 80, 80),
    12: (255, 105, 180),
    13: (139, 69, 19),
}

OCCLUSION_CLASS_IDS = {9, 10, 11, 13}
GT_COL_NAMES = [
    "frame", "id", "bb_left", "bb_top", "bb_width", "bb_height",
    "mark", "class", "visibility"
]

print("PROJECT_ROOT:", PROJECT_ROOT)
print("MOT16 exists:", MOT16_ROOT.exists())
print("GIF output folder:", OUTPUT_DIR)


PROJECT_ROOT: /home/research/tracking_crowded_people
MOT16 exists: True
GIF output folder: /home/research/tracking_crowded_people/output/jupyter-notebook


## MOT16 GT fields

Each row in `gt/gt.txt` follows:
- `frame`
- `id`
- `bb_left`
- `bb_top`
- `bb_width`
- `bb_height`
- `mark`
- `class`
- `visibility`

Important meanings:
- `id` is the tracking ID.
- `mark` is the validity flag.
- `class` includes pedestrian, occluder, crowd, reflection, and other labels.
- `visibility` shows how visible the target is.
- `frameRate` is stored in `seqinfo.ini` and is used for GIF timing.


In [3]:
# Select split/sequence and load GT annotations
DEFAULT_SPLIT = "train"
FRAME_STRIDE = 1
MAX_FRAMES_FOR_GIF = 400

split_dropdown = widgets.Dropdown(
    options=["train", "test"],
    value=DEFAULT_SPLIT,
    description="Split:",
)

sequence_dropdown = widgets.Dropdown(
    options=sorted([p.name for p in (MOT16_ROOT / DEFAULT_SPLIT).iterdir() if p.is_dir()]),
    description="Sequence:",
)

gt_view_dropdown = widgets.Dropdown(
    options=[("All GT rows", "all_gt"), ("Pedestrian only", "pedestrian_only")],
    value="all_gt",
    description="GT view:",
)

selection_output = widgets.Output()

def load_selected_sequence(*_):
    global split, SEQUENCE, GT_VIEW_MODE, seq_dir, img_dir, gt_path, gt_df, gt_plot_df, SOURCE_FPS, seqinfo_path

    split = split_dropdown.value
    SEQUENCE = sequence_dropdown.value
    GT_VIEW_MODE = gt_view_dropdown.value

    seq_dir = MOT16_ROOT / split / SEQUENCE
    img_dir = seq_dir / "img1"
    gt_path = seq_dir / "gt" / "gt.txt"
    seqinfo_path = seq_dir / "seqinfo.ini"

    assert seq_dir.exists(), f"Sequence folder not found: {seq_dir}"
    assert img_dir.exists(), f"Image folder not found: {img_dir}"
    assert gt_path.exists(), f"GT file not found: {gt_path}"
    assert seqinfo_path.exists(), f"seqinfo.ini not found: {seqinfo_path}"

    seqinfo = configparser.ConfigParser()
    seqinfo.read(seqinfo_path)
    SOURCE_FPS = seqinfo.getint("Sequence", "frameRate", fallback=30)

    gt_df = pd.read_csv(gt_path, header=None, names=GT_COL_NAMES)
    gt_df["id"] = gt_df["id"].fillna(-1).astype(int)
    gt_df["class"] = gt_df["class"].fillna(-1).astype(int)
    gt_df["mark"] = gt_df["mark"].fillna(0).astype(int)
    gt_df["visibility"] = gt_df["visibility"].fillna(0.0)

    if GT_VIEW_MODE == "pedestrian_only":
        gt_plot_df = gt_df[(gt_df["mark"] == 1) & (gt_df["class"] == 1)].copy()
    else:
        gt_plot_df = gt_df.copy()

    with selection_output:
        clear_output(wait=True)
        print(f"Using sequence: {SEQUENCE}")
        print(f"GT view mode: {GT_VIEW_MODE}")
        print(f"Sequence FPS from seqinfo.ini: {SOURCE_FPS}")
        print(f"Total GT rows loaded: {len(gt_df)}")
        print(f"Rows used for GIF: {len(gt_plot_df)}")
        positive_ids = gt_plot_df[gt_plot_df['id'] > 0]['id'].nunique()
        print(f"Unique positive IDs in view: {positive_ids}")
        display(gt_plot_df.head())

def update_sequences(change):
    selected_split = change["new"]
    sequence_dropdown.options = sorted([p.name for p in (MOT16_ROOT / selected_split).iterdir() if p.is_dir()])
    if sequence_dropdown.options:
        sequence_dropdown.value = sequence_dropdown.options[0]

split_dropdown.observe(update_sequences, names="value")
split_dropdown.observe(load_selected_sequence, names="value")
sequence_dropdown.observe(load_selected_sequence, names="value")
gt_view_dropdown.observe(load_selected_sequence, names="value")

display(split_dropdown, sequence_dropdown, gt_view_dropdown, selection_output)
load_selected_sequence()


Dropdown(description='Split:', options=('train', 'test'), value='train')

Dropdown(description='Sequence:', options=('MOT16-02', 'MOT16-04', 'MOT16-05', 'MOT16-09', 'MOT16-10', 'MOT16-…

Dropdown(description='GT view:', options=(('All GT rows', 'all_gt'), ('Pedestrian only', 'pedestrian_only')), …

Output()

## GT GIF rendering

- Each rectangle corresponds to one row from `gt/gt.txt` used by the current `GT view`.
- Box color depends on the GT class.
- Label text includes class name, ID when available, mark flag, and visibility.
- GIF timing uses the sequence FPS from `seqinfo.ini`.


In [3]:
# Draw GT annotations, export GIF, and preview it
all_frames = sorted(img_dir.glob("*.jpg"))
assert all_frames, f"No frames found in {img_dir}"

GIF_FPS = SOURCE_FPS
GIF_FRAME_DURATION = 1 / GIF_FPS

frame_numbers = [int(p.stem) for p in all_frames]
selected_numbers = frame_numbers[::FRAME_STRIDE][:MAX_FRAMES_FOR_GIF]

with Image.open(all_frames[0]) as sample_image:
    _, sample_height = sample_image.size

font_size = max(18, sample_height // 45)
font_candidates = [
    "/System/Library/Fonts/Supplemental/Arial.ttf",
    "/System/Library/Fonts/Supplemental/Arial Unicode.ttf",
    "/Library/Fonts/Arial.ttf",
    "/usr/share/fonts/truetype/dejavu/DejaVuSans-Bold.ttf",
]
font = ImageFont.load_default()
for font_path in font_candidates:
    if Path(font_path).exists():
        try:
            font = ImageFont.truetype(font_path, font_size)
            break
        except OSError:
            pass

annotated_frames = []
for frame_num in selected_numbers:
    frame_path = img_dir / f"{frame_num:06d}.jpg"
    image = Image.open(frame_path).convert("RGB")
    draw = ImageDraw.Draw(image)

    rows = gt_plot_df[gt_plot_df["frame"] == frame_num]
    for _, row in rows.iterrows():
        x1 = int(row["bb_left"])
        y1 = int(row["bb_top"])
        x2 = int(row["bb_left"] + row["bb_width"])
        y2 = int(row["bb_top"] + row["bb_height"])
        track_id = int(row["id"])
        class_id = int(row["class"])
        mark = int(row["mark"])
        visibility = float(row["visibility"])
        class_name = GT_CLASS_NAMES.get(class_id, f"class_{class_id}")
        color = GT_CLASS_COLORS.get(class_id, (255, 255, 255))

        label_parts = [class_name]
        if track_id > 0:
            label_parts.append(f"ID {track_id}")
        label_parts.append(f"m{mark}")
        label_parts.append(f"v{visibility:.2f}")
        label = " | ".join(label_parts)

        draw.rectangle([x1, y1, x2, y2], outline=color, width=4)

        text_bbox = draw.textbbox((x1, y1), label, font=font)
        text_width = text_bbox[2] - text_bbox[0]
        text_height = text_bbox[3] - text_bbox[1]
        text_x = x1 + 4
        text_y = max(0, y1 - text_height - 8)

        draw.rectangle(
            [text_x - 4, text_y - 2, text_x + text_width + 4, text_y + text_height + 2],
            fill=color,
        )
        draw.text((text_x, text_y), label, fill=(255, 255, 255), font=font)

    annotated_frames.append(image)

suffix = "all_gt_rows" if GT_VIEW_MODE == "all_gt" else "pedestrian_only"
gif_path = OUTPUT_DIR / f"{split}_{SEQUENCE}_{suffix}.gif"
imageio.mimsave(
    gif_path,
    [frame.copy() for frame in annotated_frames],
    duration=GIF_FRAME_DURATION,
    loop=0,
)

print(f"Saved GIF to: {gif_path}")
print(f"Frames in GIF: {len(annotated_frames)}")
print(f"GIF FPS from seqinfo.ini: {GIF_FPS}")
print(f"GT view mode used: {GT_VIEW_MODE}")
display(IPyImage(filename=str(gif_path)))


Saved GIF to: /Users/jainil/PycharmProjects/deep_learning_project/output/jupyter-notebook/train_MOT16-02_all_gt_rows.gif
Frames in GIF: 400
GIF FPS from seqinfo.ini: 30
GT view mode used: all_gt


## Dataset checker

This section checks every sequence in `train` and `test` for:
- frame readability
- dimension consistency
- frame count consistency
- GT presence
- positive tracking IDs
- visibility statistics
- occlusion-related rows
- class breakdowns


In [4]:
# Check every train/test sequence for frame readability, dimensions, and GT content statistics
GT_CLASS_NAMES_CHECKER = GT_CLASS_NAMES.copy()


def inspect_sequence(sequence_dir: Path, split_name: str) -> tuple[dict, list[dict]]:
    issues = []
    seqinfo_path = sequence_dir / "seqinfo.ini"
    img_dir = sequence_dir / "img1"
    gt_path = sequence_dir / "gt" / "gt.txt"

    seqinfo = configparser.ConfigParser()
    seqinfo.read(seqinfo_path)

    expected_width = None
    expected_height = None
    expected_seq_length = None
    expected_fps = None
    if seqinfo.has_section("Sequence"):
        expected_width = seqinfo.getint("Sequence", "imWidth", fallback=None)
        expected_height = seqinfo.getint("Sequence", "imHeight", fallback=None)
        expected_seq_length = seqinfo.getint("Sequence", "seqLength", fallback=None)
        expected_fps = seqinfo.getint("Sequence", "frameRate", fallback=None)

    frame_paths = sorted(img_dir.glob("*.jpg")) if img_dir.exists() else []
    unreadable_frames = []
    observed_sizes = set()

    for frame_path in frame_paths:
        try:
            with Image.open(frame_path) as img:
                img.load()
                observed_sizes.add(img.size)
        except Exception as exc:
            unreadable_frames.append(frame_path.name)
            issues.append({
                "split": split_name,
                "sequence": sequence_dir.name,
                "issue_type": "unreadable_frame",
                "details": f"{frame_path.name}: {exc}",
            })

    dominant_size = next(iter(observed_sizes)) if len(observed_sizes) == 1 else None
    dimensions_match_seqinfo = (
        dominant_size == (expected_width, expected_height)
        if dominant_size and expected_width and expected_height
        else None
    )

    frame_count_matches_seqinfo = (
        len(frame_paths) == expected_seq_length if expected_seq_length is not None else None
    )
    if frame_count_matches_seqinfo is False:
        issues.append({
            "split": split_name,
            "sequence": sequence_dir.name,
            "issue_type": "frame_count_mismatch",
            "details": f"expected {expected_seq_length}, found {len(frame_paths)}",
        })

    if len(observed_sizes) > 1:
        issues.append({
            "split": split_name,
            "sequence": sequence_dir.name,
            "issue_type": "inconsistent_dimensions",
            "details": str(sorted(observed_sizes)),
        })

    if dimensions_match_seqinfo is False:
        issues.append({
            "split": split_name,
            "sequence": sequence_dir.name,
            "issue_type": "dimension_mismatch",
            "details": f"seqinfo=({expected_width}, {expected_height}), observed={dominant_size}",
        })

    gt_exists = gt_path.exists()
    gt_rows = None
    gt_valid_rows = None
    gt_unique_ids = None
    gt_id_status = None
    avg_visibility = None
    avg_valid_visibility = None
    avg_pedestrian_visibility = None
    low_visibility_rows = None
    zero_visibility_rows = None
    occlusion_related_rows = None
    crowd_rows = None
    reflection_rows = None
    distractor_rows = None
    static_person_rows = None
    class_breakdown = None

    if gt_exists:
        gt_df_full = pd.read_csv(gt_path, header=None, names=GT_COL_NAMES)
        gt_rows = len(gt_df_full)

        gt_df_full["id"] = pd.to_numeric(gt_df_full["id"], errors="coerce").fillna(-1).astype(int)
        gt_df_full["frame"] = pd.to_numeric(gt_df_full["frame"], errors="coerce")
        gt_df_full["class"] = pd.to_numeric(gt_df_full["class"], errors="coerce").fillna(-1).astype(int)
        gt_df_full["mark"] = pd.to_numeric(gt_df_full["mark"], errors="coerce").fillna(0).astype(int)
        gt_df_full["visibility"] = pd.to_numeric(gt_df_full["visibility"], errors="coerce")

        valid_id_rows = gt_df_full[gt_df_full["id"] > 0].copy()
        gt_unique_ids = valid_id_rows["id"].nunique()
        gt_id_status = gt_unique_ids > 0
        if not gt_id_status:
            issues.append({
                "split": split_name,
                "sequence": sequence_dir.name,
                "issue_type": "missing_gt_ids",
                "details": "GT file exists but no positive tracking IDs were found.",
            })

        valid_gt_df = gt_df_full[gt_df_full["mark"] == 1].copy()
        pedestrian_df = gt_df_full[gt_df_full["class"] == 1].copy()
        gt_valid_rows = len(valid_gt_df)

        visibility_series = gt_df_full["visibility"].dropna()
        valid_visibility_series = valid_gt_df["visibility"].dropna()
        pedestrian_visibility_series = pedestrian_df["visibility"].dropna()

        avg_visibility = float(visibility_series.mean()) if not visibility_series.empty else None
        avg_valid_visibility = float(valid_visibility_series.mean()) if not valid_visibility_series.empty else None
        avg_pedestrian_visibility = float(pedestrian_visibility_series.mean()) if not pedestrian_visibility_series.empty else None
        low_visibility_rows = int((gt_df_full["visibility"].fillna(0) < 0.5).sum())
        zero_visibility_rows = int((gt_df_full["visibility"].fillna(0) <= 0).sum())

        occlusion_related_rows = int(gt_df_full["class"].isin(OCCLUSION_CLASS_IDS).sum())
        crowd_rows = int((gt_df_full["class"] == 13).sum())
        reflection_rows = int((gt_df_full["class"] == 12).sum())
        distractor_rows = int((gt_df_full["class"] == 8).sum())
        static_person_rows = int((gt_df_full["class"] == 7).sum())

        class_counts = gt_df_full["class"].value_counts().sort_index().to_dict()
        class_breakdown = {
            GT_CLASS_NAMES_CHECKER.get(class_id, f"class_{class_id}"): count
            for class_id, count in class_counts.items()
        }

        if not gt_df_full.empty:
            max_gt_frame = int(gt_df_full["frame"].max())
            min_gt_frame = int(gt_df_full["frame"].min())
            if frame_paths and (min_gt_frame < 1 or max_gt_frame > len(frame_paths)):
                issues.append({
                    "split": split_name,
                    "sequence": sequence_dir.name,
                    "issue_type": "gt_frame_out_of_range",
                    "details": f"GT frame range {min_gt_frame}-{max_gt_frame}, image frames 1-{len(frame_paths)}",
                })
    elif split_name == "train":
        issues.append({
            "split": split_name,
            "sequence": sequence_dir.name,
            "issue_type": "missing_gt",
            "details": "Training sequence has no gt/gt.txt file.",
        })

    summary_row = {
        "split": split_name,
        "sequence": sequence_dir.name,
        "fps": expected_fps,
        "num_frames": len(frame_paths),
        "unreadable_frames": len(unreadable_frames),
        "unique_dimensions": len(observed_sizes),
        "observed_dimensions": sorted(observed_sizes),
        "seqinfo_dimensions": (expected_width, expected_height),
        "dimensions_match_seqinfo": dimensions_match_seqinfo,
        "seqinfo_frame_count": expected_seq_length,
        "frame_count_matches_seqinfo": frame_count_matches_seqinfo,
        "gt_exists": gt_exists,
        "gt_rows": gt_rows,
        "gt_valid_rows": gt_valid_rows,
        "gt_has_positive_ids": gt_id_status,
        "gt_unique_ids": gt_unique_ids,
        "avg_visibility": avg_visibility,
        "avg_valid_visibility": avg_valid_visibility,
        "avg_pedestrian_visibility": avg_pedestrian_visibility,
        "low_visibility_rows": low_visibility_rows,
        "zero_visibility_rows": zero_visibility_rows,
        "occlusion_related_rows": occlusion_related_rows,
        "crowd_rows": crowd_rows,
        "reflection_rows": reflection_rows,
        "distractor_rows": distractor_rows,
        "static_person_rows": static_person_rows,
        "class_breakdown": class_breakdown,
    }
    return summary_row, issues

summary_rows = []
issue_rows = []

for split_name in ["train", "test"]:
    split_dir = MOT16_ROOT / split_name
    sequence_dirs = sorted([p for p in split_dir.iterdir() if p.is_dir()])
    for sequence_dir in sequence_dirs:
        summary_row, issues = inspect_sequence(sequence_dir, split_name)
        summary_rows.append(summary_row)
        issue_rows.extend(issues)

dataset_summary_df = pd.DataFrame(summary_rows)
dataset_issues_df = pd.DataFrame(issue_rows)

print("Dataset checker summary")
display(dataset_summary_df)

print(f"Total sequences checked: {len(dataset_summary_df)}")
print(f"Sequences with issues: {dataset_issues_df['sequence'].nunique() if not dataset_issues_df.empty else 0}")

if dataset_issues_df.empty:
    print("No dataset issues found.")
else:
    print("Dataset issues")
    display(dataset_issues_df)


Dataset checker summary


,split,sequence,fps,num_frames,unreadable_frames,unique_dimensions,observed_dimensions,seqinfo_dimensions,dimensions_match_seqinfo,seqinfo_frame_count,...,avg_valid_visibility,avg_pedestrian_visibility,low_visibility_rows,zero_visibility_rows,occlusion_related_rows,crowd_rows,reflection_rows,distractor_rows,static_person_rows,class_breakdown
0,train,MOT16-02,30,600,0,1,"[(1920, 1080)]","(1920, 1080)",True,600,...,0.417074,0.417074,18623.0,9547.0,1781.0,0.0,0.0,1200.0,5271.0,"{'pedestrian': 17833, 'person_on_vehicle': 154..."
1,train,MOT16-04,30,1050,0,1,"[(1920, 1080)]","(1920, 1080)",True,1050,...,0.617697,0.617697,33641.0,1913.0,42000.0,0.0,0.0,0.0,4798.0,"{'pedestrian': 47557, 'car': 1050, 'bicycle': ..."
2,train,MOT16-05,14,837,0,1,"[(640, 480)]","(640, 480)",True,837,...,0.520605,0.520605,3962.0,1808.0,0.0,0.0,0.0,16.0,0.0,"{'pedestrian': 6818, 'person_on_vehicle': 315,..."
3,train,MOT16-09,30,525,0,1,"[(1920, 1080)]","(1920, 1080)",True,525,...,0.589007,0.589007,3525.0,1950.0,1050.0,0.0,948.0,1575.0,0.0,"{'pedestrian': 5257, 'distractor': 1575, 'occl..."
4,train,MOT16-10,30,654,0,1,"[(1920, 1080)]","(1920, 1080)",True,654,...,0.695633,0.695633,4848.0,1436.0,2740.0,0.0,0.0,470.0,1376.0,"{'pedestrian': 12318, 'car': 25, 'static_perso..."
5,train,MOT16-11,30,900,0,1,"[(1920, 1080)]","(1920, 1080)",True,900,...,0.642527,0.642527,3368.0,1086.0,596.0,0.0,0.0,306.0,0.0,"{'pedestrian': 9174, 'distractor': 306, 'occlu..."
6,train,MOT16-13,25,750,0,1,"[(1920, 1080)]","(1920, 1080)",True,750,...,0.673300,0.673300,4268.0,243.0,3222.0,0.0,0.0,4.0,0.0,"{'pedestrian': 11450, 'car': 4484, 'bicycle': ..."
7,test,MOT16-01,30,450,0,1,"[(1920, 1080)]","(1920, 1080)",True,450,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,None
8,test,MOT16-03,30,1500,0,1,"[(1920, 1080)]","(1920, 1080)",True,1500,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,None
9,test,MOT16-06,14,1194,0,1,"[(640, 480)]","(640, 480)",True,1194,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,None


Total sequences checked: 14
Sequences with issues: 0
No dataset issues found.


## YOLO person tracking

This section runs latest Ultralytics YOLO tracking on MOT16.

Notes:
- tracker IDs here are predicted tracker IDs, not MOT16 GT IDs
- default split is `test`
- stronger defaults are used for nicer pedestrian tracking: bigger model, tuned `BoT-SORT`, ReID, GMC, and track persistence


In [5]:
# Configure higher-quality YOLO tracking with its own split/sequence selector
YOLO_MODEL_NAME = "yolo26m.pt"
YOLO_TRACKER = str(PROJECT_ROOT / "code" / "botsort_mot16_person.yaml")
YOLO_CONF = 0.25
YOLO_IOU = 0.5
YOLO_IMGSZ = 640
YOLO_PERSIST = True
YOLO_MAX_FRAMES = 300
YOLO_PERSON_ONLY = True
YOLO_DEFAULT_SPLIT = "test"

In [ ]:
# Run higher-quality YOLO tracking on the selected MOT16 sequence and export a GIF
import os
os.environ['PYTORCH_ALLOC_CONF'] = 'expandable_segments:True'

tracking_model = YOLO(YOLO_MODEL_NAME)
source_paths = sorted(yolo_img_dir.glob("*.jpg"))
assert source_paths, f"No frames found in {yolo_img_dir}"

if YOLO_MAX_FRAMES is None:
    tracking_paths = source_paths
else:
    tracking_paths = source_paths[:YOLO_MAX_FRAMES]

with Image.open(tracking_paths[0]) as sample_image:
    _, tracking_height = sample_image.size

tracking_font_size = max(20, tracking_height // 35)
tracking_font_candidates = [
    "/System/Library/Fonts/Supplemental/Arial.ttf",
    "/System/Library/Fonts/Supplemental/Arial Unicode.ttf",
    "/Library/Fonts/Arial.ttf",
    "/usr/share/fonts/truetype/dejavu/DejaVuSans-Bold.ttf",
]
tracking_font = ImageFont.load_default()
for font_path in tracking_font_candidates:
    if Path(font_path).exists():
        try:
            tracking_font = ImageFont.truetype(font_path, tracking_font_size)
            break
        except OSError:
            pass

tracking_results = tracking_model.track(
    source=[str(path) for path in tracking_paths],
    tracker=YOLO_TRACKER,
    classes=[0] if YOLO_PERSON_ONLY else None,
    conf=YOLO_CONF,
    iou=YOLO_IOU,
    imgsz=YOLO_IMGSZ,
    persist=YOLO_PERSIST,
    stream=True,
    verbose=False,
    device='0,1,2,3'
)

track_frames = []
track_counts = []
for result in tracking_results:
    frame_bgr = result.orig_img.copy()
    frame_rgb = cv2.cvtColor(frame_bgr, cv2.COLOR_BGR2RGB)
    image = Image.fromarray(frame_rgb)
    draw = ImageDraw.Draw(image)

    boxes = result.boxes
    ids = boxes.id.int().tolist() if boxes.id is not None else []
    xyxy_list = boxes.xyxy.int().tolist() if boxes.xyxy is not None else []
    track_counts.append(len(ids))

    for box_coords, track_id in zip(xyxy_list, ids):
        x1, y1, x2, y2 = box_coords
        label = f"Track {track_id}"

        draw.rectangle([x1, y1, x2, y2], outline=(255, 80, 0), width=4)
        text_bbox = draw.textbbox((x1, y1), label, font=tracking_font)
        text_width = text_bbox[2] - text_bbox[0]
        text_height = text_bbox[3] - text_bbox[1]
        text_x = x1 + 4
        text_y = max(0, y1 - text_height - 8)
        draw.rectangle(
            [text_x - 4, text_y - 2, text_x + text_width + 4, text_y + text_height + 2],
            fill=(255, 80, 0),
        )
        draw.text((text_x, text_y), label, fill=(255, 255, 255), font=tracking_font)

    track_frames.append(image)

yolo_suffix = "person_track" if YOLO_PERSON_ONLY else "all_class_track"
yolo_gif_path = OUTPUT_DIR / f"{yolo_split}_{YOLO_SEQUENCE}_{YOLO_MODEL_NAME.replace('.pt', '')}_{yolo_suffix}.gif"
imageio.mimsave(
    yolo_gif_path,
    [frame.copy() for frame in track_frames],
    duration=1 / YOLO_SOURCE_FPS,
    loop=0,
)

print(f"Saved YOLO tracking GIF to: {yolo_gif_path}")
print(f"Frames processed: {len(track_frames)}")
print(f"YOLO source FPS from seqinfo.ini: {YOLO_SOURCE_FPS}")
print(f"YOLO image size: {YOLO_IMGSZ}")
print(f"Track persistence enabled: {YOLO_PERSIST}")
print(f"Average tracks per frame: {sum(track_counts) / len(track_counts):.2f}" if track_counts else "No tracks found.")
display(IPyImage(filename=str(yolo_gif_path)))